In [ ]:
# Wrong Way Detection - Updated & Fixed Version
import os
import cv2
import math
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque

VIDEO_PATH = r"../Video_inpt/Wrong_Direction.MP4"
MODEL_PATH = r"../Models/yolo11m.pt"
OUTPUT_DIR = r"../video_out"
VIOLATIONS_DIR = os.path.join(OUTPUT_DIR, "violations")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VIOLATIONS_DIR, exist_ok=True)

model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Error opening video")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(
    os.path.join(OUTPUT_DIR, "wrong_way_output.mp4"),
    fourcc,
    fps,
    (width, height)
)

track_history = defaultdict(lambda: deque(maxlen=25))

violation_count = 0
violated_vehicles = []
violation_buffer = {}  

# تعديل عدد الفريمات المنتظرة (15 فريم تضمن استقرار البوكس وثبات اللقطة)
SAVE_DELAY_FRAMES = 15  
crossed_line_ids = set()

MIN_HISTORY = 20
MIN_MOVEMENT = 40
ANGLE_THRESHOLD = 45
DISTANCE_THRESHOLD = 120
SIZE_THRESHOLD = 50

# =========================================================
# VIOLATION LINE
# =========================================================
VIOLATION_LINE_Y = int(height * 0.65)
VEHICLE_CLASSES = [2, 3, 5, 7]

def already_counted(cx, cy, w, h):
    for car in violated_vehicles:
        distance = np.sqrt((cx - car["cx"]) ** 2 + (cy - car["cy"]) ** 2)
        if (
            distance < DISTANCE_THRESHOLD and
            abs(w - car["w"]) < SIZE_THRESHOLD and
            abs(h - car["h"]) < SIZE_THRESHOLD
        ):
            return True
    return False

# =========================================================
# MAIN PROCESSING LOOP
# =========================================================
while True:
    success, frame = cap.read()
    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )

    # قاموس لتخزين إحداثيات البوكس الحالية للسيارات في الفريم الحالي لربطها بالـ Buffer
    current_frame_boxes = {}

    if len(results) > 0:
        result = results[0]
        if result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.cpu().numpy().astype(int)
            classes = result.boxes.cls.cpu().numpy().astype(int)

            for box, track_id, cls in zip(boxes, ids, classes):
                if cls not in VEHICLE_CLASSES:
                    continue

                x1, y1, x2, y2 = map(int, box)
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)

                # حفظ الإحداثيات اللحظية للسيارة
                current_frame_boxes[track_id] = (x1, y1, x2, y2)
                track_history[track_id].append((cx, cy))

                status = "NORMAL"
                color = (0, 255, 0)
                history = track_history[track_id]

                if len(history) >= MIN_HISTORY:
                    first_x, first_y = history[0]
                    last_x, last_y = history[-1]
                    movement = abs(last_y - first_y)

                    if movement >= MIN_MOVEMENT:
                        dx = last_x - first_x
                        dy = last_y - first_y
                        angle = math.degrees(math.atan2(abs(dx), abs(dy)))

                        wrong_votes = 0
                        normal_votes = 0

                        for i in range(1, len(history)):
                            step_dy = history[i][1] - history[i - 1][1]
                            if step_dy > 0:
                                wrong_votes += 1
                            elif step_dy < 0:
                                normal_votes += 1

                        is_wrong = (
                            wrong_votes > normal_votes * 2 and
                            dy > 0 and
                            angle < ANGLE_THRESHOLD
                        )

                        if is_wrong:
                            status = "WRONG WAY"
                            color = (0, 0, 255)

                            w = x2 - x1
                            h = y2 - y1

                            if len(history) >= 2:
                                previous_y = history[-2][1]
                                current_y = history[-1][1]

                                crossed_line = (
                                    previous_y < VIOLATION_LINE_Y and
                                    current_y >= VIOLATION_LINE_Y
                                )

                                if (
                                    crossed_line and
                                    track_id not in crossed_line_ids and
                                    not already_counted(cx, cy, w, h)
                                ):
                                    # تسجيل المخالفة وبدء العداد المؤجل
                                    violation_buffer[track_id] = {
                                        "violation_number": violation_count + 1,
                                        "frame_count": 0
                                    }

                                    crossed_line_ids.add(track_id)
                                    violated_vehicles.append({
                                        "cx": cx, "cy": cy, "w": w, "h": h
                                    })
                                    violation_count += 1

                                    # قطع كادر السيارة الفوري (إذا كنت تحتاجه بجانب الفريم الكامل)
                                    crop = frame[max(0, y1):min(height, y2), max(0, x1):min(width, x2)]
                                    crop_path = os.path.join(VIOLATIONS_DIR, f"car_{violation_count}.jpg")
                                    cv2.imwrite(crop_path, crop)
                                    print(f"[VIOLATION] Vehicle {violation_count} detected.")

                # الرسم القياسي على فريم الفيديو المباشر
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"ID:{track_id} {status}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                
                pts = np.array(history, dtype=np.int32)
                if len(pts) > 1:
                    cv2.polylines(frame, [pts], False, color, 2)

    # =========================================================
    # الـ BUFFER LOOP الرئيسي (تم نقله هنا خارج الـ Tracking Loop)
    # =========================================================
    to_delete = []
    for vid, data in violation_buffer.items():
        data["frame_count"] += 1

        # إذا مر الوقت المطلوب، يتم التقاط الصورة بالفريم والبوكس الجديدين
        if data["frame_count"] >= SAVE_DELAY_FRAMES:
            vnum = data["violation_number"]
            
            # نأخذ نسخة من الفريم الحالي (الجديد والمستقر)
            frame_v = frame.copy()
            
            # إذا كانت السيارة لا تزال داخل الكادر، نرسم البوكس الحديث عليها بدقة في الصورة المحفوظة
            if vid in current_frame_boxes:
                vx1, vy1, vx2, vy2 = current_frame_boxes[vid]
                cv2.rectangle(frame_v, (vx1, vy1), (vx2, vy2), (0, 0, 255), 3)
                cv2.putText(frame_v, f"ID:{vid} WRONG WAY", (vx1, vy1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            # إضافة لوحة البيانات العلوية السوداء (Overlay)
            cv2.rectangle(frame_v, (0, 0), (width, 140), (0, 0, 0), -1)
            cv2.putText(frame_v, f"Vehicle ID: {vid}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
            cv2.putText(frame_v, "WRONG DIRECTION", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
            cv2.putText(frame_v, f"Violation Number: {vnum}", (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

            # حفظ اللقطة الكاملة والمحدثة
            save_path = os.path.join(VIOLATIONS_DIR, f"violation_{vnum}_id_{vid}.jpg")
            cv2.imwrite(save_path, frame_v)
            print(f"[SAVED] Violation {vnum} saved with stable bounding box.")
            
            to_delete.append(vid)

    for vid in to_delete:
        del violation_buffer[vid]

    # كتابة العداد الكلي على الفيديو
    cv2.putText(frame, f"Violations: {violation_count}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    video_writer.write(frame)
    cv2.imshow("Wrong Way Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        print("Stopped by user")
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()

print("\nProcessing Finished")
print(f"Violations Detected = {violation_count}")
print(f"Output Video Saved In: {OUTPUT_DIR}")
print(f"Violation Images Saved In: {VIOLATIONS_DIR}")

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


[VIOLATION] Vehicle 1 detected.
[SAVED] Violation 1 saved with stable bounding box.

Processing Finished
Violations Detected = 1
Output Video Saved In: ../video_out
Violation Images Saved In: ../video_out\violations
